# 4 · Six strategies, one fair comparison

`3_upscale_web.ipynb` made one architecture fast. Here we ask whether it is the *right*
architecture, by training several ideas with the same data, recipe and seed.

First, though, the score changes. Until now every number was RGB PSNR, which counts all
three colour channels equally. The eye sees brightness detail much better than colour
detail, so from here on the main score is **`wpsnr`**: errors in brightness (Y) weigh 6
times more than errors in colour (U, V). See `helpers/metrics.py`.

The strategies all keep the frame from notebook 3 (frozen bicubic skip, residual at low
resolution, one `PixelShuffle`) and change only the body:

1. **Control** - the 32-16 body, trained once with RGB L1 and once with a brightness-weighted L1.
2. **YUV split** - a big branch for brightness, a small one for colour.
3. **Reparameterized** - each convolution trained as three branches, merged into one after.
4. **Recursive** - one 3x3 convolution applied three times.
5. **Dense** - every layer sees all earlier layers' outputs.
6. **Attention** - a squeeze-and-excite gate on the channels.

Then the control is retrained at two more seeds, to see how much a score moves by luck.

Needs: `0_data_preprocess.ipynb`. Results: `summary.md`, "Six strategies".

In [ ]:
"""Settings and data."""

import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

from helpers import metrics, models, patches, speed, training, webexport
from helpers.training import DATA_DIR, DEVICE

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_strategies.pt"
FRAME_DIR = DATA_DIR / "synthetic"

WIDTHS = (32, 16)
STEM = 5
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0
PUBLISH = 2          # how many of the best strategies to export

training.seed_everything(SEED)
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE)
trim = training.trimmer(manifest)
val_lr, val_hr = patches.load(DATA_DIR, "val", manifest)

## 1. Why the score changed

Take one patch and damage it twice with the *same amount* of noise: once in brightness,
once in colour. RGB PSNR calls them about equal - but the colour damage is nearly
invisible. The weighted score tells them apart, like an eye does.

In [ ]:
clean = training.to_tensor(val_hr[3]).cpu()
noise = torch.randn(1, 1, *clean.shape[2:], generator=torch.Generator().manual_seed(SEED)) * 0.05
zero = torch.zeros_like(noise)
yuv = metrics.rgb_to_yuv(clean)
damaged = {
    "brightness damaged": metrics.yuv_to_rgb(yuv + torch.cat([noise, zero, zero], 1)),
    "colour damaged": metrics.yuv_to_rgb(yuv + torch.cat([zero, noise, zero], 1)),
}
metrics.print_table({name: metrics.report(image, clean) for name, image in damaged.items()})

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, (name, image) in zip(axes, {"clean": clean, **damaged}.items()):
    ax.imshow(image[0].clamp(0, 1).permute(1, 2, 0).numpy(), interpolation="nearest")
    ax.set_title(name)
    ax.axis("off")
fig.tight_layout()
plt.show()

There are two ways to combine three channels into one number:

- `wpsnr` weights the **errors** 6:1:1, then converts to dB. This is what we report.
- `wpsnr_db` averages the three **dB numbers** 6:1:1 - the convention in papers. One
  very accurate channel lifts it more.

`chroma_subsample=2` scores colour at the resolution 4:2:0 video actually sends.

In [ ]:
print(f"{'':<20}{'wpsnr':>8}{'wpsnr_db':>10}{'wpsnr 4:2:0':>13}{'rgb':>8}")
for name, image in damaged.items():
    print(f"{name:<20}{metrics.wpsnr(image, clean):8.2f}{metrics.wpsnr_db(image, clean):10.2f}"
          f"{metrics.wpsnr(image, clean, chroma_subsample=2):13.2f}{metrics.psnr(image, clean):8.2f}")

## 2. The strategies

The control, `models.PlainUpscaler`, and the reparameterized one, `models.ReparamUpscaler`,
live in `helpers/models.py` because later notebooks use them too. The rest are defined
here. Each one only builds its layers and a `residual(x)`; `models.Upscaler` adds the
bicubic skip and the `PixelShuffle`.

In [ ]:
"""Strategy 2: split brightness and colour."""

def frozen_conv(weight):
    """A 1x1 convolution with fixed weights (out, in), not trained."""
    conv = nn.Conv2d(weight.shape[1], weight.shape[0], 1, bias=False)
    with torch.no_grad():
        conv.weight.copy_(weight[:, :, None, None])
    conv.weight.requires_grad_(False)
    return conv


def colour_matrix(matrix, phases):
    """A 3x3 colour matrix applied to each of `phases` interleaved copies of the channels."""
    weight = torch.zeros(3 * phases, 3 * phases)
    for out_channel in range(3):
        for in_channel in range(3):
            for phase in range(phases):
                weight[out_channel * phases + phase, in_channel * phases + phase] = matrix[out_channel, in_channel]
    return weight


def plain_body(widths, in_channels, stem):
    """stem x stem, then 3x3s, ReLU after each."""
    layers = []
    for index, width in enumerate(widths):
        kernel = stem if index == 0 else 3
        layers += [nn.Conv2d(in_channels, width, kernel, padding=kernel // 2), nn.ReLU(inplace=True)]
        in_channels = width
    return nn.Sequential(*layers), in_channels


class YuvSplit(models.Upscaler):
    """Convert to Y'CbCr, a wide branch for Y (brightness), a narrow one for U and V.

    `luma_sees=1`: the Y branch reads Y only. `luma_sees=3`: it reads Y, U and V.
    Y is taken with a 1x1 convolution rather than `x[:, :1]`, because a slice exports as
    `Slice`, which is outside the browser's fast operators.
    """

    def __init__(self, luma=(32, 16), chroma=(8, 4), luma_sees=1, stem=STEM, scale=2):
        super().__init__(scale)
        phases = scale ** 2
        self.luma_sees = luma_sees
        self.to_luma = frozen_conv(metrics.RGB_TO_YUV[:1])
        self.to_yuv = frozen_conv(metrics.RGB_TO_YUV)
        self.luma_body, luma_width = plain_body(luma, luma_sees, stem)
        self.luma_tail = models.zero_start(nn.Conv2d(luma_width, phases, 3, padding=1))
        self.chroma_body, chroma_width = plain_body(chroma, 3, stem)
        self.chroma_tail = models.zero_start(nn.Conv2d(chroma_width, 2 * phases, 3, padding=1))
        self.to_rgb = frozen_conv(colour_matrix(metrics.YUV_TO_RGB, phases))
        self.halo = self.halo_for(stem, max(len(luma), len(chroma)))

    def residual(self, x):
        yuv = self.to_yuv(x)
        luma = self.luma_tail(self.luma_body(yuv if self.luma_sees == 3 else self.to_luma(x)))
        chroma = self.chroma_tail(self.chroma_body(yuv))
        return self.to_rgb(torch.cat([luma, chroma], dim=1))

In [ ]:
"""Strategies 4, 5 and 6."""

class Recursive(models.Upscaler):
    """One shared 3x3 applied `steps` times: depth without extra parameters."""

    def __init__(self, width=16, steps=3, stem=STEM, scale=2):
        super().__init__(scale)
        self.steps = steps
        self.stem = nn.Conv2d(3, width, stem, padding=stem // 2)
        self.cell = nn.Conv2d(width, width, 3, padding=1)
        self.tail = models.zero_start(nn.Conv2d(width, 3 * scale ** 2, 3, padding=1))
        self.halo = self.halo_for(stem, steps + 1)

    def residual(self, x):
        h = F.relu(self.stem(x))
        for _ in range(self.steps):
            h = F.relu(self.cell(h)) + h
        return self.tail(h)


class Dense(models.Upscaler):
    """Every layer reads all earlier outputs, joined with Concat."""

    def __init__(self, width=14, growth=12, layers=2, stem=STEM, scale=2):
        super().__init__(scale)
        self.stem = nn.Conv2d(3, width, stem, padding=stem // 2)
        self.layers = nn.ModuleList([nn.Conv2d(width + i * growth, growth, 3, padding=1) for i in range(layers)])
        self.tail = models.zero_start(nn.Conv2d(width + layers * growth, 3 * scale ** 2, 3, padding=1))
        self.halo = self.halo_for(stem, layers + 1)

    def residual(self, x):
        features = [F.relu(self.stem(x))]
        for layer in self.layers:
            features.append(F.relu(layer(torch.cat(features, dim=1))))
        return self.tail(torch.cat(features, dim=1))


class Attention(models.Upscaler):
    """The 32-16 body with a squeeze-and-excite gate: each channel is scaled by a weight
    computed from the average of the whole tile. (So the output depends on the whole tile,
    and the halo no longer covers everything - tiles would not join seamlessly.)"""

    def __init__(self, widths=WIDTHS, stem=STEM, squeeze=4, scale=2):
        super().__init__(scale)
        self.body, width = plain_body(widths, 3, stem)
        self.gate = nn.Sequential(nn.Conv2d(widths[0], widths[0] // squeeze, 1), nn.ReLU(),
                                  nn.Conv2d(widths[0] // squeeze, widths[0], 1), nn.Sigmoid())
        self.tail = models.zero_start(nn.Conv2d(width, 3 * scale ** 2, 3, padding=1))
        self.halo = self.halo_for(stem, len(widths))

    def residual(self, x):
        wide = self.body[1](self.body[0](x))
        gated = wide * self.gate(wide.mean(dim=(2, 3), keepdim=True))
        return self.tail(self.body[3](self.body[2](gated)))

In [ ]:
"""Every run: name -> (model class, arguments, loss)."""

STRATEGIES = {
    "32-16 rgb l1": (models.PlainUpscaler, {}, F.l1_loss),
    "32-16 luma l1": (models.PlainUpscaler, {}, metrics.luma_chroma_l1),
    "yuv split": (YuvSplit, {}, metrics.luma_chroma_l1),
    "reparameterized": (models.ReparamUpscaler, {}, metrics.luma_chroma_l1),
    "recursive x3": (Recursive, {}, metrics.luma_chroma_l1),
    "dense cascade": (Dense, {}, metrics.luma_chroma_l1),
    "attention (se)": (Attention, {}, metrics.luma_chroma_l1),
    "yuv split, wide": (YuvSplit, {"luma_sees": 3}, metrics.luma_chroma_l1),
}

### Cost and halo of each

- **MACs per LR pixel** (multiply-adds) is what a tile costs to run.
- **halo** is how far a model reaches. Each class declares it; we also *measure* it: set
  every weight to a small positive number, send a gradient back from one output pixel,
  and see how far into the input it arrives. A wrong halo would show seams between tiles.

In [ ]:
def measured_halo(model, size=33):
    """How far one output pixel reaches into the input, in LR pixels."""
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, nn.Conv2d):
                module.weight.fill_(0.05)
                if module.bias is not None:
                    module.bias.zero_()
    x = torch.ones(1, 3, size, size, requires_grad=True)
    centre = size // 2
    model(x)[0, :, centre * SCALE, centre * SCALE].sum().backward()
    reach = x.grad.abs().sum(dim=(0, 1))
    rows = torch.nonzero(reach.sum(dim=1)).flatten()
    cols = torch.nonzero(reach.sum(dim=0)).flatten()
    return int(max(centre - rows.min(), rows.max() - centre, centre - cols.min(), cols.max() - centre))


print(f"{'strategy':<18}{'params':>9}{'MACs/LR px':>12}{'halo':>6}{'measured':>12}")
built = {}
for name, (build, arguments, _) in STRATEGIES.items():
    torch.manual_seed(SEED)
    built[name] = build(scale=SCALE, **arguments).to(DEVICE)
    reach = measured_halo(build(scale=SCALE, **arguments))
    whole_tile = reach >= 33 // 2
    assert whole_tile or reach == built[name].halo, f"{name}: declares {built[name].halo}, reaches {reach}"
    print(f"{name:<18}{models.count_parameters(built[name]):9,}{speed.macs_per_lr_pixel(built[name]):12,.0f}"
          f"{built[name].halo:6}{'whole tile' if whole_tile else reach:>12}")

## 3. Train every strategy

One recipe for all: 100 epochs, Adam at 1e-3 with a cosine schedule, batch 32, the same
seed. The best epoch by `wpsnr` is kept. This takes a while.

In [ ]:
torch.manual_seed(SEED)
baseline = training.evaluate(models.Resize(SCALE).to(DEVICE), val_loader, trim)

results, histories = {"bicubic": baseline}, {}
for name, (_, _, loss_fn) in STRATEGIES.items():
    started = time.time()
    histories[name], best = training.train(built[name], train_loader, val_loader, trim, EPOCHS,
                                           learning_rate=LEARNING_RATE, loss_fn=loss_fn, print_every=0)
    results[name] = training.evaluate(built[name], val_loader, trim)
    print(f"{name:<18} best epoch {best['epoch']:3}, {time.time() - started:5.0f}s")

order = sorted(STRATEGIES, key=lambda name: -results[name]["wpsnr"])
best_name = order[0]
control = results["32-16 rgb l1"]["wpsnr"]
metrics.print_table({name: results[name] for name in ["bicubic", *order]},
                    extra=lambda name, s: f"{s['wpsnr'] - control:+8.2f} vs control")
print(f"\nbest: {best_name}")

### How big must a difference be?

The validation set is small, so scores move a little by chance. To see how much, train
the *same* control twice more with other seeds. A gap smaller than that spread is luck.

In [ ]:
repeats = {"seed 0": results["32-16 rgb l1"]}
for seed in (SEED + 1, SEED + 2):
    torch.manual_seed(seed)
    again = models.PlainUpscaler(scale=SCALE).to(DEVICE)
    training.train(again, train_loader, val_loader, trim, EPOCHS, learning_rate=LEARNING_RATE, print_every=0)
    repeats[f"seed {seed}"] = training.evaluate(again, val_loader, trim)
metrics.print_table(repeats)

spread = max(s["wpsnr"] for s in repeats.values()) - min(s["wpsnr"] for s in repeats.values())
beaten = [name for name in order[1:] if results[best_name]["wpsnr"] - results[name]["wpsnr"] > spread]
print(f"\nthe same model at three seeds: {spread:.2f} dB apart")
print(f"clearly beaten by {best_name}: {', '.join(beaten) or 'none'}")

In [ ]:
"""Training curves, and quality against cost."""

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for name in STRATEGIES:
    left.plot(range(1, EPOCHS + 1), [h["wpsnr"] for h in histories[name]], label=name, linewidth=1.2)
    cost = speed.macs_per_lr_pixel(built[name])
    right.scatter(cost, results[name]["wpsnr"], s=28)
    right.annotate(name, (cost, results[name]["wpsnr"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
for ax in (left, right):
    ax.axhline(baseline["wpsnr"], color="0.6", linestyle="--", linewidth=1)
left.set_title("validation wpsnr (dB)")
left.set_xlabel("epoch")
left.legend(fontsize=7)
right.set_title("quality against cost")
right.set_xlabel("MACs per LR pixel (while training)")
fig.tight_layout()
plt.show()

### The reparameterized model, merged

Its three branches per layer must merge into one convolution without changing anything.

In [ ]:
merged = built["reparameterized"].eval().collapsed()
training.assert_same_model(built["reparameterized"], merged)
results["reparameterized (collapsed)"] = training.evaluate(merged, val_loader, trim)
built["reparameterized (collapsed)"] = merged
for name in ("reparameterized", "reparameterized (collapsed)"):
    print(f"{name:<28}{models.count_parameters(built[name]):8,} params"
          f"{speed.macs_per_lr_pixel(built[name]):8,.0f} MACs{results[name]['wpsnr']:8.2f} dB")

### Where the errors are

Brightness errors sit on edges and are clearly visible; colour errors are a faint wash.

In [ ]:
@torch.no_grad()
def error_panels(model, index=3):
    lr, hr = training.to_tensor(val_lr[index]), training.to_tensor(val_hr[index])
    pictures = {"target": hr, "bicubic": trim(models.bicubic(lr, SCALE)), best_name: trim(model.eval()(lr))}
    fig, axes = plt.subplots(3, 3, figsize=(10, 9.6))
    for column, (name, picture) in enumerate(pictures.items()):
        picture = picture.clamp(0, 1)
        axes[0][column].imshow(picture[0].permute(1, 2, 0).cpu().numpy(), interpolation="nearest")
        axes[0][column].set_title(name)
        error = (metrics.rgb_to_yuv(picture) - metrics.rgb_to_yuv(hr))[0].cpu()
        for row, (label, plane) in enumerate([("brightness error", error[0].abs()),
                                              ("colour error", error[1:].abs().sum(0))], start=1):
            axes[row][column].imshow(plane.numpy(), cmap="inferno", vmin=0, vmax=0.15)
            axes[row][column].set_title(f"{label}  {plane.mean():.4f}", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


error_panels(built[best_name])

## 4. Save and export

The best strategy is saved and reloaded. Then the best `PUBLISH` architectures are
exported to the benchmark page at float32 and float16 (the reparameterized one in its
merged form, which is what would ship).

In [ ]:
BUILDERS = {"PlainUpscaler": models.PlainUpscaler, "ReparamUpscaler": models.ReparamUpscaler,
            "YuvSplit": YuvSplit, "Recursive": Recursive, "Dense": Dense, "Attention": Attention}

winner = built[best_name]
training.save_checkpoint(CHECKPOINT, winner, architecture=type(winner).__name__,
                         arguments=STRATEGIES[best_name][1], strategy=best_name, scale=SCALE,
                         halo=winner.halo, epochs=EPOCHS, val=results[best_name], val_bicubic=baseline,
                         degradation=manifest["degradation"])
checkpoint = training.load_checkpoint(CHECKPOINT)
restored = BUILDERS[checkpoint["architecture"]](scale=SCALE, **checkpoint["arguments"]).to(DEVICE)
restored.load_state_dict(checkpoint["state_dict"])
training.assert_same_model(winner, restored)
print("reloaded model is identical")

In [ ]:
published, classes = [], set()
for name in order:  # one per model class, best first
    if STRATEGIES[name][0] not in classes:
        classes.add(STRATEGIES[name][0])
        published.append("reparameterized (collapsed)" if name == "reparameterized" else name)
    if len(published) == PUBLISH:
        break

for name in published:
    model = built[name]
    slug = name.replace(" (collapsed)", "").replace(" ", "_")
    size = webexport.TILE_STEP + 2 * model.halo
    noise = np.random.rand(1, 3, size, size).astype(np.float32)
    fp32 = webexport.export(model, f"upscale_{slug}_tile{webexport.TILE_STEP}.onnx", size,
                            label=f"{name} fp32", halo=model.halo)
    fp16 = webexport.to_float16(fp32, f"upscale_{slug}_fp16_tile{webexport.TILE_STEP}.onnx")
    webexport.write_metadata(fp16["path"], {"label": f"{name} fp16"})
    for info, dtype, tolerance in [(fp32, np.float32, 2e-3), (fp16, np.float16, 4e-3)]:
        webexport.check_graph(info["path"], model, noise, dtype, tolerance)
        ms = webexport.time_graph(info["path"], size, dtype)
        webexport.annotate(info, cpu_ms=round(ms, 2))
        print(webexport.summary_line(info, ms))

In [ ]:
"""A whole frame through the winner."""

truth = Image.open(sorted(FRAME_DIR.glob("*.png"))[0]).convert("RGB")
truth = truth.crop((0, 0, truth.width - truth.width % SCALE, truth.height - truth.height % SCALE))
small = training.to_tensor(truth.resize((truth.width // SCALE, truth.height // SCALE), Image.BICUBIC))
with torch.no_grad():
    upscaled = restored.eval()(small)
target = training.to_tensor(truth)
metrics.print_table({"bicubic": metrics.report(models.bicubic(small, SCALE), target),
                     best_name: metrics.report(upscaled, target)})
Image.fromarray(training.to_image(upscaled)).save(training.SAMPLE_DIR / "upscaled_strategies.png")